MLBD Assignment - November 2025 - Cranfield University

This programme will perform some computations on a dataset containing the daily new cases of COVID-19 registered in 201 locations from 22/01/2020 to 09/03/2023 using Spark. This dataset can be obtained here: __[COIVD-19 confirmed cases dataset](https://github.com/CSSEGISandData/COVID-19/blob/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv)__. <br>

The computations are divided into different queries:<br>

- Initialisation: Spark session initialisation, data loading and dataset transformation.<br>
- Query 1: for each country, the mean number of daily confirmed cases for each month in the dataset will be computed.<br>
- Query 2: for each continent, the mean, standard deviation, minimum and maximum of the daily confirmed cases for each week, considering only the 100 most affected states by the pandemic according to the trendline coefficient will be computed.<br>
- Query 3: for each month, the 50 most affected states monthly according to the trendline coefficient will be segregated into 4 groups using the clustering algorithm K-means.<br>

In addition to that, some results and graphs will be saved into the system.<br>

The results are not printed due to their magnitude, please refer to the results directory created when running the programme to check them.

In [1]:
# Imports and definitions of constants

# Imports of modules used throughout the programme
import numpy as np
import time
import random
import os
import matplotlib.pyplot as plt
import seaborn as sns

import findspark
findspark.init()

import pyspark
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark import SparkFiles
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.sql.functions import col, expr, mean, stddev, lag, to_date, day, weekofyear, month, year, date_format, \
                                  coalesce, lit, datediff, date_trunc, mean as _mean, stddev as _stddev, \
                                  count as _count, sum as _sum, min as _min, max as _max, trim, desc
from pyspark.sql.window import Window

# Search/Creation of the directories to save the results

cwd = os.getcwd()

results_dir = os.path.join(cwd, "results")
os.makedirs(results_dir, exist_ok=True)
q1_dir = os.path.join(results_dir, "Query 1")
os.makedirs(q1_dir, exist_ok=True)
q1g_dir = os.path.join(q1_dir, "graphs")
os.makedirs(q1g_dir, exist_ok=True)
q2_dir = os.path.join(results_dir, "Query 2")
os.makedirs(q2_dir, exist_ok=True)
q2g_dir = os.path.join(q2_dir, "graphs")
os.makedirs(q2g_dir, exist_ok=True)
q3_dir = os.path.join(results_dir, "Query 3")
os.makedirs(q3_dir, exist_ok=True)
spark_dir = os.path.join(q3_dir, "spark")
os.makedirs(spark_dir, exist_ok=True)
naive_dir = os.path.join(q3_dir, "naive")
os.makedirs(naive_dir, exist_ok=True)
sparkg_dir = os.path.join(spark_dir, "graphs")
os.makedirs(sparkg_dir, exist_ok=True)
naiveg_dir = os.path.join(naive_dir, "graphs")
os.makedirs(naiveg_dir, exist_ok=True)

# Spark Session initialisation

s_t0 = time.time() # Get time before Spark Session initialisation

spark = SparkSession.builder.appName('COVID_Analysis').master('local').getOrCreate()

s_t1 = time.time() # Get time after Spark Session initialisation
t_spark_session_init = s_t1 - s_t0  # Computation of the processing time for initialising the Spark Session

print(f"Processing time for initialising the Spark Session: {t_spark_session_init} s")

Processing time for initialising the Spark Session: 33.279513359069824 s


In [2]:
# Data loading

d_t0 = time.time() # Get time before data loading

spark.sparkContext.addFile("time_series_covid19_confirmed_global.csv")
df = spark.read.csv(SparkFiles.get("time_series_covid19_confirmed_global.csv"), header=True, inferSchema=True)

d_t1 = time.time() # Get time after data loading
t_load_data = d_t1 - d_t0 # Computation of the processing time for loading the data

print(f"Processing time for loading the data: {t_load_data} s")

Processing time for loading the data: 8.697003841400146 s


In [3]:
# Dataset transformation

td_t0 = time.time() # Get time before dataset transformation

# Date columns identification
date_cols = df.columns[4:]

# Format change from wide to long so that there is just one column for dates and confirmed cases.
unpivot_expr = "stack({0}, {1}) as (Date, Confirmed)".format(
    len(date_cols),
    ",".join([f"'{c}', `{c}`" for c in date_cols])
)

df_long = df.selectExpr(
    "`Province/State`",
    "`Country/Region`",
    "Lat",
    "Long",
    unpivot_expr
)

# Type conversion
df_long = df_long.withColumn("Confirmed", col("Confirmed").cast("int")) \
                 .withColumn("Date", to_date(col("Date"), "M/d/yy"))

# Daily new cases per Country/Region computation 
# (Confirmed - Prev_Confirmed, being Prev_Confirmed the same column as Confirmed but with a lag of one day)
window_spec = Window.partitionBy("Country/Region", "Province/State").orderBy("Date")

df_daily = df_long.withColumn("Prev_Confirmed", lag("Confirmed").over(window_spec)) \
                  .withColumn("Daily_New", expr("Confirmed - Prev_Confirmed")) \
                  .fillna(0, subset=["Daily_New"]) # Replace nan with 0 to avoid computation errors

# Date-related columns addition in an isolated way to perform some computations
data = df_daily.withColumn("Year", year("Date")) \
               .withColumn("Week", weekofyear("Date")) \
               .withColumn("Month", month("Date")) \
               .withColumn("Day", day("Date")) \
               .withColumn("YearMonth", date_format("Date", "yyyy-MM"))

# Numeric day index column creation for regression
data = data.withColumn("day_idx", datediff(col("Date"), lit("2020-01-22")))

# StateKey column creation. StateKey = Province/State if present, else Country/Region
data = data.withColumn(
    "StateKey",
    coalesce(col("Province/State"), col("Country/Region"))
)

td_t1 = time.time() # Get time after dataset transformation
t_dataset_transformation = td_t1 - td_t0 # Computation of the processing time for transforming the dataset

print(f"Processing time for transforming the dataset: {t_dataset_transformation} s")

Processing time for transforming the dataset: 1.6156673431396484 s


In [4]:
############
# QUERY 1  #
############

# Montly mean computation of daily confirmed cases for each country

m_t0 = time.time() # Get time before mean computation

monthly_mean = data.groupBy("Country/Region", "YearMonth") \
                            .agg(mean("Daily_New").alias("mean_daily_cases")) \
                            .orderBy("Country/Region", "YearMonth")

m_t1 = time.time() # Get time after mean computation
t_mean = m_t1 - m_t0 # Computation of the processing time for computing the mean (Query 1)
print(f"Processing time for computing the mean (Query 1): {t_mean} s")

Processing time for computing the mean (Query 1): 0.2302534580230713 s


In [5]:
############
# QUERY 2  #
############

# Weekly statistics computation of daily confirmed cases for each continent, 
# considering only the 100 most affected states by the pandemic according to the trendline coefficient

tc_t0 = time.time()  # Get time before trendline coefficient computation

# Parameter computation to obtain the trendline coefficient (slope of the linear regression) per state.
# As stated in the report, slope = (sum_xy - n*mean_x*mean_y) / (sum_x2 - n*mean_x^2), where
# x = day index and y = daily confirmed cases
agg = data.groupBy("StateKey").agg(
    _count("Daily_New").alias("n"),
    _mean("day_idx").alias("mean_x"),
    _mean("Daily_New").alias("mean_y"),
    _sum(expr("CAST(day_idx AS DOUBLE) * CAST(Daily_New AS DOUBLE)")).alias("sum_xy"), # Cast to avoid overflow
    _sum(expr("CAST(day_idx AS DOUBLE) * CAST(day_idx AS DOUBLE)")).alias("sum_x2") # Cast to avoid overflow
)

# Trendline coefficient computation
agg = agg.withColumn(
    "denom", expr("sum_x2 - n * mean_x * mean_x")
).withColumn(
    "numer", expr("sum_xy - n * mean_x * mean_y")
).withColumn(
    "slope",
    expr("CASE WHEN denom = 0 THEN 0 ELSE numer / denom END") # Avoid computation error when dividing by 0
).select("StateKey", "slope")

tc_t1 = time.time() # Get time after trendline coefficient computation
t_trendline_coeff = tc_t1 - tc_t0 # Computation of the processing time for computing the trendline coefficient

t100_t0 = time.time() # Get time before top 100 selection

# Top 100 states selection by slope (trendline coefficient)
TOP_N = 100
top_states = agg.orderBy(col("slope").desc()).limit(TOP_N)
top_state_keys = [r["StateKey"] for r in top_states.select("StateKey").collect()]

# Data filtering to top 100 states
df_top100 = data.filter(col("StateKey").isin(top_state_keys))

t100_t1 = time.time() # Get time after top 100 selection
t_top100 = t100_t1 - t100_t0 # Computation of the processing time for selecting the 100 most affected states

c_t0 = time.time() # Get time before continent mapping

# Continent mapping CSV loading
continent_raw = spark.read.option("header", True).option("sep", ";").csv("countries_continents.csv") # Separator is ; due to system language being Spanish
csv_cols = continent_raw.columns

country_col, continent_col = csv_cols[0], csv_cols[1]

continent_df = continent_raw.withColumnRenamed("StateKey", "Country/Region") \
                            .withColumnRenamed(continent_col, "Continent") \
                            .select("Country/Region", "Continent")

# Whitespace normalisation in Country/Region to improve join matching
continent_df = continent_df.withColumn("Country/Region", trim(col("Country/Region")))

# Continent info joining
df_top100 = df_top100.join(continent_df, on="Country/Region", how="left") \
                     .withColumn("Continent", coalesce(col("Continent"), lit("Unknown")))

# Weekly statistics computation of daily confirmed cases for each continent
df_top100 = df_top100.withColumn("week_start", date_trunc("week", col("Date")).cast("date")) # Start of the week column addition

c_t1 = time.time() # Get time after continent mapping and week column addition
t_continent = c_t1 - c_t0 # Computation of the processing time for adding the continent and start of the week column

ws_t0 = time.time() # Get time before weekly statistics computation

weekly_stats_by_continent = df_top100.groupBy("Continent", "week_start").agg(
    _mean("Daily_New").alias("mean_daily_cases"),
    _stddev("Daily_New").alias("stddev_daily_cases"),
    _min("Daily_New").alias("min_daily_cases"),
    _max("Daily_New").alias("max_daily_cases")
).orderBy("Continent", "week_start")

ws_t1 = time.time() # Get time after weekly statistics computation
t_weekly_stats = ws_t1 - ws_t0 # Computation of the processing time for computing the weekly statistics
t_total2 = ws_t1 - tc_t0 # Computation of the processing time for query 2

print(f"Processing time for computing the trendline coefficient: {t_trendline_coeff} s")
print(f"Processing time for selecting the top 100 states: {t_top100} s")
print(f"Processing time for adding the continent and start of the week column: {t_continent} s")
print(f"Processing time for computing the weekly statistics: {t_weekly_stats} s")
print(f"Total processing time for query 2: {t_total2} s")

Processing time for computing the trendline coefficient: 0.3078916072845459 s
Processing time for selecting the top 100 states: 10.020263195037842 s
Processing time for adding the continent and start of the week column: 0.5154314041137695 s
Processing time for computing the weekly statistics: 0.096771240234375 s
Total processing time for query 2: 10.940357446670532 s


In [6]:
############
# QUERY 3  #
############

# Segregation for each month of the 50 most affected states monthly according to the trendline coefficient 
# into 4 groups using the clustering algorithm K-means.


mtc_t0 = time.time() # Get time before monthly trendline coefficient computation

# Parameter computation to obtain the trendline coefficient (slope of the linear regression) per state per month.
# As stated in the report, slope = (sum_xy - n*mean_x*mean_y) / (sum_x2 - n*mean_x^2), where
# x = day index and y = daily confirmed cases
agg_monthly = data.groupBy("YearMonth", "StateKey").agg(
    _count("Daily_New").alias("n"),
    _mean("day_idx").alias("mean_x"),
    _mean("Daily_New").alias("mean_y"),
    _sum(expr("CAST(day_idx AS DOUBLE) * CAST(Daily_New AS DOUBLE)")).alias("sum_xy"), # Cast to avoid overflow
    _sum(expr("CAST(day_idx AS DOUBLE) * CAST(day_idx AS DOUBLE)")).alias("sum_x2") # Cast to avoid overflow
)

# Monthly trendline coefficient computation
agg_monthly = agg_monthly.withColumn(
    "denom", expr("sum_x2 - n * mean_x * mean_x")
).withColumn(
    "numer", expr("sum_xy - n * mean_x * mean_y")
).withColumn(
    "slope",
    expr("CASE WHEN denom = 0 THEN 0 ELSE numer / denom END") # Avoid computation error when dividing by 0
).select("StateKey", "slope")

mtc_t1 = time.time() # Get time after monthly trendline coefficient computation
t_monthly_trendline_coeff = mtc_t1 - mtc_t0 # Computation of the processing time for computing the monthly trendline coefficient

print(f"Processing time for computing the monthly trendline coefficient: {t_mean} s")

Processing time for computing the monthly trendline coefficient: 0.2302534580230713 s


In [7]:
# SPARK IMPLEMENTATION

# K-means algorithm computation for each month under the required conditions

# Months collection
months = [row["YearMonth"] for row in data.select("YearMonth").distinct().orderBy("YearMonth").collect()]
print(f"Found {len(months)} months. Processing each month with top-50 states and Spark ML library's KMeans(k=4)...\n")

# For each month, top 50 states selection, K-means and silhouette computation, store the results and graphs 

results_summary_spark = [] # List to store the results for the summary

tcs_t0 = time.time() # Get time before Spark's K-means implementation

clusters_spark_dir = os.path.join(q3_dir, "clusters_spark.txt") # File creation to store the cluster list
with open(clusters_spark_dir, "w") as file:
    for m in months:
        print(f"Processing month {m}")
        file.write(f"\n==== MONTH: {m} ====")
        
        cs_t0 = time.time() # Get time before a single Spark's K-means implementation

        # Top 50 states selection
        
        # Month filtering
        month_df = agg_monthly.filter(col("YearMonth") == lit(m))
        
        # Top 50 states selection by slope (trendline coefficient)
        top50 = month_df.sort(desc("slope")).limit(50)

        # Data verification to avoid computation errors
        count_top = top50.count()
        if count_top == 0:
            print(f"No rows for month {m}, skipping.")
            continue
        if count_top < 4:
            print(f"Only {count_top} records for month {m}. KMeans with k=4 may fail; skipping.")
            continue

        #K-means
        
        # Features assembling (slope)
        feature_cols = ["slope"]
        assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
        top50_vec = assembler.transform(top50).select("StateKey", "slope", "features")
    
        # K-means fitting (K=4)
        k = 4
        kmeans = KMeans(k=k, seed=42, featuresCol="features", predictionCol="cluster")
        model = kmeans.fit(top50_vec)
        transformed = model.transform(top50_vec).select("StateKey", "slope", "cluster")
    
        # Silhouette computation with PySpark evaluator
        evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="cluster", metricName="silhouette", distanceMeasure="squaredEuclidean")
        top50_vec_with_pred = model.transform(assembler.transform(top50)).select(col("StateKey"), col("slope"), col("features"), col("cluster"))
        try:
            spark_silhouette = evaluator.evaluate(top50_vec_with_pred)
        except Exception as e:
            spark_silhouette = float('nan')
    
        # Data saving in CSV format
        pdf = top50_vec_with_pred.toPandas().rename(columns={"StateKey": "state", "slope": "slope", "cluster": "cluster"})
        pdf_dir = os.path.join(spark_dir, m+".csv")
        pdf.to_csv(pdf_dir, index=False)
        
        # Cluster centers and boundaries extraction
        centers = [float(c[0]) for c in model.clusterCenters()]
        boundaries = [(sorted(centers)[i] + sorted(centers)[i+1]) / 2 for i in range(len(centers)-1)]
    
        cs_t1 = time.time() # Get time after a single Spark's K-means implementation
        t_cluster_spark = cs_t1 - cs_t0 # Computation of the processing time for a single Spark's K-means implementation
        
        file.write(f"\nProcessing time for computing the clusters: {t_cluster_spark} s")
    
        # Cluster membership lists saving in a text file
        clusters = {}
        for ci in range(k):
            members = pdf.loc[pdf['cluster'] == ci, 'state'].tolist()
            clusters[ci] = members
            file.write(f"\nCluster {ci} (n={len(members)}):")
            file.write(", ".join(str(x) for x in members))
    
        file.write(f"\nSilhouette (PySpark evaluator): {spark_silhouette:.4f}") # Silhouette coefficient saving in a text file
    
        # Summary saving
        results_summary_spark.append({
            'month': m,
            'kmeans_model': model,
            'spark_silhouette': spark_silhouette,
            'pdf': pdf,
            'clusters': clusters,
            'time': t_cluster_spark
        })
        
        # Plotting
        plt.figure(figsize=(12, 6))
        pal = sns.color_palette("tab10", k)
    
        # Data plotting
        sns.scatterplot(x="slope", y=np.zeros(len(pdf)), hue="cluster", data=pdf,
                        palette=pal, s=120, alpha=0.8, edgecolor="black")
    
        # Cluster centers & boundaries plotting
        cl = [0,1,2,3]
        for c in centers:
            l = plt.axvline(c, color=pal[centers.index(c)], linestyle="--", alpha=0.7)
            cl[sorted(centers).index(c)] = l.get_color()
        for b in boundaries:
            plt.axvline(b, color=cl[boundaries.index(b)], linestyle="-", alpha=0.7)
            plt.axvline(b, color=cl[boundaries.index(b)+1], linestyle="--", alpha=0.7)
    
        plt.title(f"K-Means Clustering of Top 50 States — {m}\nSilhouette Score: {spark_silhouette:.3f}")
        plt.xlabel("Trendline slope (KMeans feature space)")
        plt.ylabel("")
        plt.yticks([])
        plt.legend(title="Cluster")
        plt.grid(True, axis="x", linestyle="--", alpha=0.3)
        plt.tight_layout()

        # Plot saving
        graph_dir = os.path.join(sparkg_dir, m)
        plt.savefig(graph_dir, dpi=300, bbox_inches='tight')
        plt.close("all") # Avoid graph plotting
        
file.close

tcs_t1 = time.time() # Get time after Spark's K-means implementation
t_total_clusters_spark = tcs_t1 - tcs_t0 # Computation of the processing time for Spark's K-means implementation

print(f"\n\nTotal processing time for computing the clusters (Spark): {t_total_clusters_spark} s")

# Summary saving in a text file
sparktxt_dir = os.path.join(q3_dir, "clusters_spark_summary.txt")
with open(sparktxt_dir, "w") as file:
    for r in results_summary_spark:
        file.write(f"Month: {r['month']}, spark_silhouette = {r['spark_silhouette']:.4f}, processing time = {r['time']} s\n")
file.close

Found 39 months. Processing each month with top-50 states and Spark ML library's KMeans(k=4)...

Processing month 2020-01
Processing month 2020-02
Processing month 2020-03
Processing month 2020-04
Processing month 2020-05
Processing month 2020-06
Processing month 2020-07
Processing month 2020-08
Processing month 2020-09
Processing month 2020-10
Processing month 2020-11
Processing month 2020-12
Processing month 2021-01
Processing month 2021-02
Processing month 2021-03
Processing month 2021-04
Processing month 2021-05
Processing month 2021-06
Processing month 2021-07
Processing month 2021-08
Processing month 2021-09
Processing month 2021-10
Processing month 2021-11
Processing month 2021-12
Processing month 2022-01
Processing month 2022-02
Processing month 2022-03
Processing month 2022-04
Processing month 2022-05
Processing month 2022-06
Processing month 2022-07
Processing month 2022-08
Processing month 2022-09
Processing month 2022-10
Processing month 2022-11
Processing month 2022-12
Pro

<function TextIOWrapper.close()>

In [8]:
# NAÏVE IMPLEMENTATION

# K-means algorithm computation for each month under the required conditions

k = 4
max_iters = 100
tolerance = 1e-4
random.seed(42)

# For each month, top 50 states selection, K-means and silhouette computation, store the results and graphs 

results_summary_naive = [] # List to store the results for the summary

tcn_t0 = time.time() # Get time before Spark's K-means implementation

clusters_naive_dir = os.path.join(q3_dir, "clusters_naive.txt") # File creation to store the cluster list
with open(clusters_naive_dir, "w") as file:
    for m in months:
        print(f"Processing month {m}")
        file.write(f"\n==== MONTH: {m} ====")
    
        cn_t0 = time.time() # Get time before a single naïve K-means implementation
        
        # Top 50 states selection
        
        # Month filtering
        month_df = agg_monthly.filter(col("YearMonth") == lit(m))
        
        # Top 50 states selection by slope (trendline coefficient)
        top50 = month_df.sort(desc("slope")).limit(50)
        pdf = top50.select("StateKey", "slope").toPandas()
        pdf.columns = ["state", "slope"]

        # Data verification to avoid computation errors
        if len(pdf) < k:
            print(f"Skipping month {m}: only {len(pdf)} rows.")
            continue
    
        X = pdf["slope"].values.reshape(-1, 1)
    
        # K-means
        # Centroids initialisation (random points)
        centroids = random.sample(list(X.flatten()), k)
        centroids = np.array(sorted(centroids))
    
        for it in range(max_iters):
            # Point assignation to nearest centroid
            distances = np.abs(X - centroids.reshape(1, -1))
            cluster_labels = np.argmin(distances, axis=1)
    
            # New centroids computation
            new_centroids = np.array([X[cluster_labels == j].mean() if np.any(cluster_labels == j) else centroids[j]
                                      for j in range(k)])
    
            # Convergence check
            shift = np.linalg.norm(new_centroids - centroids)
            centroids = new_centroids
            if shift < tolerance:
                break

        # Cluster centers and boundaries extraction
        pdf["cluster"] = cluster_labels
        centers = [float(c) for c in sorted(centroids)]
        boundaries = [(sorted(centers)[i] + sorted(centers)[i+1]) / 2 for i in range(len(centers)-1)]

        # Data saving in CSV format
        pdf_dir = os.path.join(naive_dir, m+".csv")
        pdf.to_csv(pdf_dir, index=False)
    
        # Silhouette coefficient
        cluster_means = [np.mean(X[cluster_labels == i]) for i in range(k)]
        inter_dists = [abs(cluster_means[i] - cluster_means[j]) for i in range(k) for j in range(i+1, k)]
        silhouette_val = np.mean(inter_dists) / abs(np.max(inter_dists)) if abs(np.max(inter_dists)) > 0 else 0.0
    
        cn_t1 = time.time() # Get time after a single naïve K-means implementation
        t_clusters_naive = cn_t1 - cn_t0 # Computation of the processing time for a single naïve K-means implementation
    
        # Summary saving
        results_summary_naive.append({
            'month': m,
            'naive_silhouette': silhouette_val,
            'pdf': pdf,
            'time': t_clusters_naive
        })
    
        # Cluster membership lists and additional data saving in a text file
        file.write(f"\nProcessing time for computing the clusters: {t_clusters_naive} s")
        file.write(f"\nNaive silhouette-like separation score: {silhouette_val:.4f}")
        for cluster_id in range(k):
            members = pdf.loc[pdf["cluster"] == cluster_id, "state"].tolist()
            file.write(f"\nCluster {cluster_id} (n={len(members)}):")
            file.write(", ".join(members))
    
        # Plotting
        plt.figure(figsize=(12, 6))
        pal = sns.color_palette("tab10", k)
    
        # Data plotting
        sns.scatterplot(x="slope", y=np.zeros(len(pdf)), hue="cluster", data=pdf,
                        palette=pal, s=120, alpha=0.8, edgecolor="black")
    
        # Cluster centers & boundaries plotting
        cl = [0,1,2,3]
        for c in centers:
            l = plt.axvline(c, color=pal[centers.index(c)], linestyle="--", alpha=0.7)
            cl[sorted(centers).index(c)] = l.get_color()
        for b in boundaries:
            plt.axvline(b, color=cl[boundaries.index(b)], linestyle="-", alpha=0.7)
            plt.axvline(b, color=cl[boundaries.index(b)+1], linestyle="--", alpha=0.7)
    
        plt.title(f"K-Means Clustering of Top 50 States — {m}\nSilhouette Score: {silhouette_val:.3f}")
        plt.xlabel("Trendline slope (KMeans feature space)")
        plt.ylabel("")
        plt.yticks([])
        plt.legend(title="Cluster")
        plt.grid(True, axis="x", linestyle="--", alpha=0.3)
        plt.tight_layout()

        # Plot saving
        graph_dir = os.path.join(naiveg_dir, m)
        plt.savefig(graph_dir, dpi=300, bbox_inches='tight')
        plt.close("all") # Avoid graph plotting

file.close

tcn_t1 = time.time() # Get time after naïve K-means implementation
t_total_clusters_naive = tcn_t1 - tcn_t0 # Computation of the processing time for the naïve K-means implementation

print(f"\n\nTotal processing time for computing the clusters (Naive): {t_total_clusters_naive} s")

# Summary saving in a text file
naivetxt_dir = os.path.join(q3_dir, "clusters_naive_summary.txt")
with open(naivetxt_dir, "w") as file:
    for r in results_summary_naive:
        file.write(f"Month: {r['month']}, naive_silhouette = {r['naive_silhouette']:.4f}, processing time = {r['time']} s\n")
file.close

Processing month 2020-01
Processing month 2020-02
Processing month 2020-03
Processing month 2020-04
Processing month 2020-05
Processing month 2020-06
Processing month 2020-07
Processing month 2020-08
Processing month 2020-09
Processing month 2020-10
Processing month 2020-11
Processing month 2020-12
Processing month 2021-01
Processing month 2021-02
Processing month 2021-03
Processing month 2021-04
Processing month 2021-05
Processing month 2021-06
Processing month 2021-07
Processing month 2021-08
Processing month 2021-09
Processing month 2021-10
Processing month 2021-11
Processing month 2021-12
Processing month 2022-01
Processing month 2022-02
Processing month 2022-03
Processing month 2022-04
Processing month 2022-05
Processing month 2022-06
Processing month 2022-07
Processing month 2022-08
Processing month 2022-09
Processing month 2022-10
Processing month 2022-11
Processing month 2022-12
Processing month 2023-01
Processing month 2023-02
Processing month 2023-03


Total processing time f

<function TextIOWrapper.close()>

In [9]:
# Results saving from queries 1 and 2

# QUERY 1

# Long Formatted
monthly_mean_long_dir = os.path.join(q1_dir, "monthly_mean_long.csv")
monthly_mean_long_pd = monthly_mean.toPandas()
monthly_mean_long_pd.to_csv(monthly_mean_long_dir, index=False)

# Wide Formatted
monthly_mean_wide = monthly_mean.groupBy("Country/Region").pivot("YearMonth").avg("mean_daily_cases").orderBy("Country/Region")
monthly_mean_wide_dir = os.path.join(q1_dir, "monthly_mean_wide.csv")
monthly_mean_wide_pd = monthly_mean_wide.toPandas()
monthly_mean_wide_pd.to_csv(monthly_mean_wide_dir, index=False)

# QUERY 2

# Long Formatted
weekly_stats_by_continent_long_dir = os.path.join(q2_dir, "weekly_stats_by_continent_long.csv")
weekly_stats_by_continent_pd = weekly_stats_by_continent.toPandas()
weekly_stats_by_continent_pd.to_csv(weekly_stats_by_continent_long_dir, index=False)

# Wide Formatted
# Mean
weekly_mean_wide = weekly_stats_by_continent.groupBy("Continent").pivot("week_start").avg("mean_daily_cases").orderBy("Continent")
weekly_mean_by_continent_wide_dir = os.path.join(q2_dir, "weekly_mean_by_continent_wide.csv")
weekly_mean_wide_pd = weekly_mean_wide.toPandas()
weekly_mean_wide_pd.to_csv(weekly_mean_by_continent_wide_dir, index=False)

# Standard Deviation
weekly_stddev_wide = weekly_stats_by_continent.groupBy("Continent").pivot("week_start").avg("stddev_daily_cases").orderBy("Continent")
weekly_stddev_by_continent_wide_dir = os.path.join(q2_dir, "weekly_stddev_by_continent_wide.csv")
weekly_stddev_wide_pd = weekly_stddev_wide.toPandas()
weekly_stddev_wide_pd.to_csv(weekly_stddev_by_continent_wide_dir, index=False)

# Minimum
weekly_min_wide = weekly_stats_by_continent.groupBy("Continent").pivot("week_start").avg("min_daily_cases").orderBy("Continent")
weekly_min_by_continent_wide_dir = os.path.join(q2_dir, "weekly_min_by_continent_wide.csv")
weekly_min_wide_pd = weekly_min_wide.toPandas()
weekly_min_wide_pd.to_csv(weekly_min_by_continent_wide_dir, index=False)

# Maximum
weekly_max_wide = weekly_stats_by_continent.groupBy("Continent").pivot("week_start").avg("max_daily_cases").orderBy("Continent")
weekly_max_by_continent_wide_dir = os.path.join(q2_dir, "weekly_max_by_continent_wide.csv")
weekly_max_wide_pd = weekly_max_wide.toPandas()
weekly_max_wide_pd.to_csv(weekly_max_by_continent_wide_dir, index=False)

# Processing time results
time_dir = os.path.join(results_dir, "time.txt")
with open(time_dir, "w") as file:
    file.write(f"Processing time for initialising the Spark Session: {t_spark_session_init} s\n")
    file.write(f"Processing time for loading the data: {t_load_data} s\n")
    file.write(f"Processing time for transforming the dataset: {t_dataset_transformation} s\n")
    file.write(f"Processing time for computing the mean (Query 1): {t_mean} s\n")
    file.write(f"Processing time for computing the trendline coefficient: {t_trendline_coeff} s\n")
    file.write(f"Processing time for selecting the top 100 states: {t_top100} s\n")
    file.write(f"Processing time for adding the continent and start of the week column: {t_continent} s\n")
    file.write(f"Processing time for computing the weekly statistics: {t_weekly_stats} s\n")
    file.write(f"Total processing time for query 2: {t_total2} s\n")
    file.write(f"Processing time for computing the monthly trendline coefficient: {t_mean} s\n")
    file.write(f"Total processing time for computing the clusters (Spark): {t_total_clusters_spark} s\n")
    file.write(f"Total processing time for computing the clusters (Naive): {t_total_clusters_naive} s\n")
file.close

<function TextIOWrapper.close()>

In [ ]:
# Graphs saving for queries 1 and 2

# QUERY 1

# Monthly mean plot
monthly_mean_wide_pd_2 = monthly_mean_wide_pd.set_index("Country/Region")
monthly_mean_wide_pd_2.T.plot(figsize=(12, 6))
plt.title(f"Mean Monthly Daily Cases by Country")
plt.xlabel("Month")
plt.ylabel("Mean Daily Cases")
plt.legend().remove()
plt.grid(True, axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
graph_dir = os.path.join(q1g_dir, "monthly_mean")
plt.savefig(graph_dir, dpi=300, bbox_inches='tight')
plt.close("all")

# Monthly mean plot without the top 3 countries
monthly_mean_wide_pd_3 = monthly_mean_wide_pd.set_index("Country/Region").drop("US").drop("India").drop("Korea, South")
monthly_mean_wide_pd_3.T.plot(figsize=(12, 6))
plt.title(f"Mean Monthly Daily Cases by Country (without top 3 countries)")
plt.xlabel("Month")
plt.ylabel("Mean Daily Cases")
plt.legend().remove()
plt.grid(True, axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
graph_dir = os.path.join(q1g_dir, "monthly_mean_without_top3")
plt.savefig(graph_dir, dpi=300, bbox_inches='tight')
plt.close("all")

# Maximum mean monthly daily cases histogram
# All the countries
max_values = monthly_mean_wide_pd_2.T.max()
plt.figure(figsize=(10, 5))
counts, bins, patches = plt.hist(max_values, bins=20, edgecolor='k')
plt.title("Histogram of Maximum Mean Monthly Daily Cases")
plt.xlabel("Max Monthly Mean Value")
plt.ylabel("Frequency")

# Add count labels
for count, xleft, xright in zip(counts, bins[:-1], bins[1:]):
    if count > 0:
        x = (xleft + xright) / 2
        plt.text(
            x,
            count,
            str(int(count)),
            ha='center',
            va='bottom',
            fontsize=9
        )
plt.tight_layout()
graph_dir = os.path.join(q1g_dir, "monthly_mean_histogram")
plt.savefig(graph_dir, dpi=300, bbox_inches='tight')
plt.close("all")

# Countries below the 33000 maximum daily cases
max_values_2 = max_values[max_values < 33000]
plt.figure(figsize=(10, 5))
counts, bins, patches = plt.hist(max_values_2, bins=20, edgecolor='k')
plt.title("Histogram of Maximum Mean Monthly Daily Cases below 33000")
plt.xlabel("Max Monthly Mean Value")
plt.ylabel("Frequency")

# Add count labels
for count, xleft, xright in zip(counts, bins[:-1], bins[1:]):
    if count > 0:
        x = (xleft + xright) / 2
        plt.text(
            x,
            count,
            str(int(count)),
            ha='center',
            va='bottom',
            fontsize=9
        )
plt.tight_layout()
graph_dir = os.path.join(q1g_dir, "monthly_mean_histogram_below_33000")
plt.savefig(graph_dir, dpi=300, bbox_inches='tight')
plt.close("all")

# QUERY 2

# Weekly mean daily cases plot 
weekly_mean_wide_pd_2 = weekly_mean_wide_pd.set_index("Continent")
weekly_mean_wide_pd_2.T.plot(figsize=(12, 6))
plt.title(f"Mean Weekly Daily Cases by Continent")
plt.xlabel("Week")
plt.ylabel("Mean Daily Cases")
plt.grid(True, axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
graph_dir = os.path.join(q2g_dir, "weekly_mean")
plt.savefig(graph_dir, dpi=300, bbox_inches='tight')
plt.close("all")

# Weekly standard deviation of daily cases plot 
weekly_stddev_wide_pd_2 = weekly_stddev_wide_pd.set_index("Continent")
weekly_stddev_wide_pd_2.T.plot(figsize=(12, 6))
plt.title(f"Standard Deviation of Weekly Daily Cases by Continent")
plt.xlabel("Week")
plt.ylabel("Standard Deviation of Daily Cases")
plt.grid(True, axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
graph_dir = os.path.join(q2g_dir, "weekly_stddev")
plt.savefig(graph_dir, dpi=300, bbox_inches='tight')
plt.close("all")

# Weekly minimum daily cases plot 
weekly_min_wide_pd_2 = weekly_min_wide_pd.set_index("Continent")
weekly_min_wide_pd_2.T.plot(figsize=(12, 6))
plt.title(f"Minumum Weekly Daily Cases by Continent")
plt.xlabel("Week")
plt.ylabel("Minimum Daily Cases")
plt.grid(True, axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
graph_dir = os.path.join(q2g_dir, "weekly_min")
plt.savefig(graph_dir, dpi=300, bbox_inches='tight')
plt.close("all")

# Weekly maximum daily cases plot 
weekly_max_wide_pd_2 = weekly_max_wide_pd.set_index("Continent")
weekly_max_wide_pd_2.T.plot(figsize=(12, 6))
plt.title(f"Maximum Weekly Daily Cases by Continent")
plt.xlabel("Week")
plt.ylabel("Maximum Daily Cases")
plt.grid(True, axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
graph_dir = os.path.join(q2g_dir, "weekly_max")
plt.savefig(graph_dir, dpi=300, bbox_inches='tight')
plt.close("all")

In [ ]:
# Spark session termination

spark.stop()